In [3]:
# Transformers example -> Sorting a sequence of numbers


In [4]:
import flax.nnx as nnx
import jax
import jax.numpy as jnp
import jax.random as jrandom
import optax

from probjax.nn.nets.transformer import PosEmbed, Transformer, LearnedPosEmbed

In [5]:
VOCAB_SIZE = 10

In [6]:
def generate_data(key, n, T, vocab_size=10):
    sequences = jrandom.randint(key, (n, T,1), 0, vocab_size, dtype=jnp.int32)

    sequences_sorted = jnp.sort(sequences, axis=-2)

    return sequences, sequences_sorted

inputs, labels = generate_data(jax.random.PRNGKey(0), 1000, 10, VOCAB_SIZE)

2024-10-25 15:29:34.305747: W external/xla/xla/service/gpu/nvptx_compiler.cc:930] The NVIDIA driver's CUDA version is 12.5 which is older than the PTX compiler version 12.6.77. Because the driver is older than the PTX compiler version, XLA is disabling parallel compilation, which may slow down compilation. You should update your NVIDIA driver or use the NVIDIA-provided CUDA forward compatibility packages.


In [7]:
key = jrandom.PRNGKey(0)

In [51]:
class Model(nnx.Module, experimental_pytree=True):

    def __init__(self, dim,rngs, dropout_rate=0.1):
        self.embed = nnx.Embed(VOCAB_SIZE, dim, rngs=rngs)
        self.pos_embed = PosEmbed(dim, 100,rngs=rngs)
        self.transformer = Transformer(dim, 1,4,10, widening_factor=10,rngs=rngs, dropout_rate=dropout_rate)
        self.output = nnx.Linear(dim, VOCAB_SIZE, rngs=rngs)

    def __call__(self, x, deterministic=False):
        x = self.embed(x)
        x = jnp.squeeze(x,axis=-2)
        x = self.pos_embed(x)
        x = self.transformer(x,deterministic)
        x = self.output(x)
        return x



In [52]:
model = Model(50, rngs=nnx.Rngs(0), dropout_rate=None)

In [53]:
nnx.display(model)

In [54]:
params = nnx.state(model, nnx.Param)

In [55]:
optimizer = optax.adam(5e-4)
opt_state = optimizer.init(params)

In [72]:

train_seq_len = [10,100]
def loss_fn(params,model, key):
    nnx.update(model, params)
    l = 0
    for t in train_seq_len:
        key, key_sub = jax.random.split(key)
        inp_data, labels = generate_data(key_sub, 128, t, vocab_size=VOCAB_SIZE)
        logits = model(inp_data)
        labels = jnp.squeeze(jax.nn.one_hot(labels, VOCAB_SIZE), -2)
        loss = optax.softmax_cross_entropy(logits, labels).mean()
        l += loss
    return l

@jax.jit
def acc(params,model, inputs, outputs):
    nnx.update(model, params)
    inp_data, labels = inputs, outputs
    logits = model(inp_data)
    labels = jnp.squeeze(jax.nn.one_hot(labels, VOCAB_SIZE), -2)
    acc = (logits.argmax(axis=-1) == labels.argmax(-1)).mean()
    return acc

@jax.jit
def update(params,model, key, opt_state):
    loss, grads = jax.value_and_grad(loss_fn)(params,model, key)
    updates, opt_state = optimizer.update(grads, opt_state)
    params = optax.apply_updates(params, updates)
    return loss, params, opt_state

In [73]:
model.train()

In [74]:
key = jrandom.PRNGKey(0)

In [75]:

for i in range(10_000):
    key, subkey, key2 = jrandom.split(key, 3)
    loss, params, opt_state = update(params,model,key, opt_state)
    if (i % 1000) == 0:
        inputs, labels = generate_data(key, 32,100, vocab_size=VOCAB_SIZE)
        accuracy = acc(params,model, inputs, labels)
        print(accuracy, loss)

0.95437497 16.371952
0.96999997 0.36469823
0.983125 0.11847164
0.98281246 0.059861116
0.953125 0.23275895
0.9975 0.025078867
0.99625 0.021202805
0.99562496 0.023352962
0.87031245 0.94442403
0.99875 0.01151962


In [76]:
model.eval()
nnx.update(model, params)

In [77]:
input = jax.random.randint(key+1, (1, 10,1),0, 10,dtype=jnp.int32)
outputs = model(input)
print(input[0,...,0])
print(outputs.argmax(-1)[0])
print(jnp.sort(input[0,...,0]))

[0 9 7 8 0 7 0 9 0 1]
[0 0 0 0 1 7 7 8 9 9]
[0 0 0 0 1 7 7 8 9 9]


In [71]:
jnp.allclose(outputs.argmax(-1)[0], jnp.sort(input[0,...,0]))

Array(False, dtype=bool)

In [ ]:
input = jax.random.randint(key, (1, 10,1),0, 10,dtype=jnp.int32)
outputs = f.apply(params, key + 2, input)
print(input[0,...,0])
print(outputs.argmax(-1)[0])

[0 0 2 1 9 1 3 0 6 5]
[0 0 0 1 1 2 3 5 6 9]
